# Benchmarking clustering algorithms

PCD file is uploaded trhough the upload file widget.

Benchmatking settings allow user to set minimum, maximum and step when benchmarking. Setting maximum to 0 will mean all points are included in analysis. Fot this project, analysis was limited to 50,000 to reduce time intensity.


In [1]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from html import escape
import time
from sklearn.cluster import DBSCAN, KMeans, SpectralClustering
import hdbscan


## Upload file and preform initial data processing

In [2]:


def _pcd_bytes_to_xyz(raw_bytes):
    raw_bytes = bytes(raw_bytes)
    lines = raw_bytes.decode("utf-8", errors="ignore").splitlines()
    fields = None
    data_start = None
    data_mode = None
    for index, line in enumerate(lines):
        parts = line.strip().split()
        if not parts or parts[0].startswith("#"):
            continue
        if parts[0].upper() == "FIELDS":
            fields = parts[1:]
        elif parts[0].upper() == "DATA":
            data_mode = parts[1].lower() if len(parts) > 1 else None
            data_start = index + 1
            break

    # raise error if file contains non ascii charachters
    if data_start is None or data_mode != "ascii":
        raise ValueError("This app supports ASCII PCD files only.")

    # splic by commas or spaces
    rows = [[float(value) for value in line.replace(",", " ").split()] for line in lines[data_start:] if line.strip() and not line.strip().startswith("#")]
    data = np.asarray(rows, dtype=float)

    # check the file contains data
    if data.size == 0:
        raise ValueError("The PCD file contains no point data.")

    # reshape rows to columns
    if data.ndim == 1:
        data = data.reshape(1, -1)
    columns = [fields.index(axis) for axis in ("x", "y", "z")] if fields and all(axis in fields for axis in ("x", "y", "z")) else [0, 1, 2]

    # raise erroe of file does not contain all coordinates
    if data.shape[1] < 3:
        raise ValueError("The PCD file must contain x, y and z data.")
    return data[:, columns]


In [3]:
# Allow for manual filtering by distance from the sensor before analysis. 
def filter(points, threshold):
    """ Reduce number of points by distance from sensor"""
    distances = np.linalg.norm(points, axis=1)
    threshold_number = threshold * distances.max()
    return points[distances <= threshold_number]


In [4]:
def _on_upload(change):
    if not upload.value:
        return
    uploaded = next(iter(upload.value.values())) if isinstance(upload.value, dict) else upload.value[0]
    try:
        app_state["raw_points"] = _pcd_bytes_to_xyz(uploaded["content"])
        status.value = f"Loaded {len(app_state['raw_points']):,} points from the uploaded file."
    except Exception as error:
        app_state["raw_points"] = None
        status.value = f'<span style="color:#a11">{escape(str(error))}</span>'


In [5]:
app_state = {"raw_points": None}
upload = widgets.FileUpload(accept=".pcd", multiple=False, description="Upload PCD")
status = widgets.HTML(value="Upload an ASCII PCD file to begin.")
filter_threshold = widgets.FloatSlider(
    value=1.0,
    min=0.01,
    max=1.0,
    step=0.01,
    description="Distance limit",
    readout_format=".0%",
    continuous_update=False,
)


In [6]:
upload.observe(_on_upload, names="value")
display(widgets.VBox([
    upload,
    status,
    widgets.HTML("<b>Distance filtering</b>"),
    filter_threshold,
]))
print("Upload a point cloud to configure the timing benchmark.")


Upload a point cloud to configure the timing benchmark.


In [7]:
n_clusters = widgets.IntSlider(
    value=10,
    min=2,
    max=20,
    description="Max clusters (KMeans/Spectral)",
    continuous_update=False,
)
dbscan_eps = widgets.FloatLogSlider(
    value=0.05,
    base=10,
    min=-3,
    max=0,
    step=0.05,
    description="DBSCAN eps",
    continuous_update=False,
)
min_samples = widgets.IntSlider(
    value=10,
    min=2,
    max=100,
    description="Min samples",
    continuous_update=False,
)
display(widgets.VBox([
    widgets.HTML("<b>Algorithm parameters</b>"),
    n_clusters,
    dbscan_eps,
    min_samples,
]))


## Set up benchmarking and run analysis


In [8]:
benchmark_start_points = widgets.IntText(value=200, description="Start points")
benchmark_step_points = widgets.IntText(value=2000, description="Step points")
benchmark_max_points = widgets.IntText(value=0, description="Max points (0 = all)")
benchmark_run_button = widgets.Button(
    description="Benchmark algorithms",
    button_style="primary",
    icon="clock",
)
benchmark_output = widgets.Output()
benchmark_results = []


def _run_algorithm_benchmark(_):
    benchmark_run_button.disabled = True
    try:
        with benchmark_output:
            clear_output(wait=True)
            if app_state["raw_points"] is None or not len(app_state["raw_points"]):
                raise ValueError("Upload a PCD file before running the benchmark.")
            if benchmark_start_points.value < 2:
                raise ValueError("Start points must be at least 2.")
            if benchmark_step_points.value < 1:
                raise ValueError("Step points must be at least 1.")
            if benchmark_max_points.value < 0:
                raise ValueError("Maximum points cannot be negative; use 0 for all available points.")

            # Use range-filtered uploaded data directly; manual boxes and annotation labels are not used.
            benchmark_points = filter(app_state["raw_points"], filter_threshold.value)
            if len(benchmark_points) < 2:
                raise ValueError("Fewer than two points remain after the distance filter.")

            maximum = (
                len(benchmark_points)
                if benchmark_max_points.value == 0
                
                else min(benchmark_max_points.value, len(benchmark_points))
            )
            start = min(benchmark_start_points.value, maximum)
            if maximum < benchmark_start_points.value:
                print(
                    f"Only {maximum:,} distance-filtered points are available; "
                    f"starting at {start:,} points."
                )
            sample_sizes = list(range(start, maximum + 1, benchmark_step_points.value))
            if not sample_sizes or sample_sizes[-1] != maximum:
                sample_sizes.append(maximum)

            # Every algorithm sees the same nested samples at each point count.
            random_seed = 0
            permutation = np.random.default_rng(random_seed).permutation(len(benchmark_points))
            sample_pool = benchmark_points[permutation]
            algorithm_names = ("K-means", "DBSCAN", "HDBSCAN", "Spectral")
            timing_rows = []
            benchmark_results.clear()
            print(
                f"Benchmarking {len(sample_sizes)} sample sizes from "
                f"{len(benchmark_points):,} points after distance filtering. "
                "Times include fit_predict only."
            )

            # Set up clustering algorithms
            for point_count in sample_sizes:
                sample = sample_pool[:point_count]
                estimators = {
                    "K-means": KMeans(
                        n_clusters=min(n_clusters.value, point_count),
                        n_init=10,
                        random_state=42,
                    ),
                    "DBSCAN": DBSCAN(
                        eps=dbscan_eps.value,
                        min_samples=min(min_samples.value, point_count),
                    ),
                    "HDBSCAN": hdbscan.HDBSCAN(
                        min_cluster_size=min(min_samples.value, point_count),
                        min_samples=min(min_samples.value, point_count),
                    ),
                    "Spectral": SpectralClustering(
                        n_clusters=min(n_clusters.value, point_count - 1),
                        affinity="nearest_neighbors",
                        n_neighbors=min(10, point_count - 1),
                        assign_labels="kmeans",
                        random_state=42,
                    ),
                }


                # Print status
                for name in algorithm_names:
                    started = time.perf_counter()
                    try:
                        estimators[name].fit_predict(sample)
                    except Exception as error:
                        elapsed = time.perf_counter() - started
                        timing_rows.append({
                            "algorithm": name,
                            "point_count": point_count,
                            "seconds": elapsed,
                            "error": str(error),
                        })
                        print(f"{name} at {point_count:,} points failed after {elapsed:.3f}s: {error}")
                        continue
                    elapsed = time.perf_counter() - started
                    timing_rows.append({
                        "algorithm": name,
                        "point_count": point_count,
                        "seconds": elapsed,
                        "error": None,
                    })
                    benchmark_results.append({
                        "algorithm": name,
                        "point_count": point_count,
                        "seconds": elapsed,
                    })
                    print(f"{name}: {point_count:,} points in {elapsed:.4f}s")

            # Plot graph
            figure, axes = plt.subplots(figsize=(10, 6))
            colors = plt.get_cmap("tab10")
            for algorithm_index, name in enumerate(algorithm_names):
                successful = [row for row in benchmark_results if row["algorithm"] == name]
                if successful:
                    axes.plot(
                        [row["point_count"] for row in successful],
                        [row["seconds"] for row in successful],
                        marker="o",
                        label=name,
                        color=colors(algorithm_index),
                    )
            axes.set_title("Clustering fit time vs sample size")
            axes.set_xlabel("Number of points in sample")
            axes.set_ylabel("Fit time (seconds)")
            axes.grid(True, alpha=0.3)
            axes.legend()
            figure.tight_layout()
            display(figure)
            plt.close(figure)

            headers = ("Algorithm", "Sample points", "Fit time (s)", "Status")
            table = "<table><tr>" + "".join(
                f"<th style='padding:4px 10px'>{header}</th>" for header in headers
            ) + "</tr>"
            table += "".join(
                "<tr>" + "".join(
                    f"<td style='padding:4px 10px'>{value}</td>"
                    for value in (
                        row["algorithm"],
                        f"{row['point_count']:,}",
                        f"{row['seconds']:.4f}",
                        row["error"] or "OK",
                    )
                ) + "</tr>"
                for row in timing_rows
            )
            table += "</table>"
            display(widgets.HTML(value=table))
    except Exception as error:
        with benchmark_output:
            print(f"Benchmark failed: {error}")
    finally:
        benchmark_run_button.disabled = False


benchmark_run_button.on_click(_run_algorithm_benchmark)
display(widgets.VBox([
    widgets.HTML(
        "<b>Clustering time vs sample size</b><br>"
    ),
    widgets.HBox([
        benchmark_start_points,
        benchmark_step_points,
        benchmark_max_points,
    ]),
    benchmark_run_button,
    benchmark_output,
]))
